# Extracción y Análisis de Datos de Reddit sobre Memecoins  
**Objetivo**: Extraer posts de subreddits de criptomonedas (DOGE, SHIB, PEPE) y analizar su relación con la volatilidad.  
**Fuente**: API de Reddit via PRAW.

### 1. Instalación de dependencias

In [57]:
!pip install praw pandas matplotlib tqdm psaw
import praw
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import time
import re
import requests
from collections import defaultdict
from tqdm import tqdm
print("Librerías instaladas y cargadas")

Librerías instaladas y cargadas


### 2. Parámetros y función de extracción

In [55]:
SUBREDDITS = ["CryptoMarkets", "CryptoCurrency", "CryptoNews"]
MAX_POSTS_PER_DAY = 5
DAYS_BACK = 365

def get_pushshift_posts(subreddit, date):
    after = int(datetime.combine(date, datetime.min.time()).timestamp())
    before = int(datetime.combine(date + timedelta(days=1), datetime.min.time()).timestamp())

    url = "https://api.pushshift.io/reddit/search/submission/"
    params = {
        "subreddit": subreddit,
        "after": after,
        "before": before,
        "size": MAX_POSTS_PER_DAY,
        "sort": "asc"
    }

    try:
        response = requests.get(url, params=params)
        data = response.json().get("data", [])
    except Exception as e:
        print(f"Error en {subreddit} ({date}): {e}")
        return []

    if len(data) < MAX_POSTS_PER_DAY:
        print(f"Solo se encontraron {len(data)} posts en r/{subreddit} el {date}")


    posts = []
    for post in data:
        posts.append({
            "id": post.get("id"),
            "subreddit": subreddit,
            "title": post.get("title", ""),
            "text": post.get("selftext", ""),
            "score": post.get("score", 0),
            "upvote_ratio": post.get("upvote_ratio", None),
            "num_comments": post.get("num_comments", 0),
            "author": post.get("author", ""),
            "created_utc": datetime.utcfromtimestamp(post.get("created_utc")).strftime('%Y-%m-%d %H:%M:%S'),
            "url": f"https://reddit.com{post.get('permalink', '')}"
        })
    return posts

### 3. Extracción de datos

In [56]:
from tqdm import tqdm
from datetime import datetime, timedelta

all_posts = []
today = datetime.utcnow().date()
dates = [today - timedelta(days=i) for i in range(DAYS_BACK)]

print(f"Iniciando extracción de {MAX_POSTS_PER_DAY} posts/día × {len(SUBREDDITS)} subreddits × {DAYS_BACK} días")

for date in tqdm(dates, desc="Procesando fechas"):
    for subreddit in SUBREDDITS:
        daily_posts = get_pushshift_posts(subreddit, date)
        all_posts.extend(daily_posts)

    if (dates.index(date) + 1) % 30 == 0:
        print(f"Procesados {dates.index(date)+1} días hasta la fecha: {date}")

# Guardar CSV
import pandas as pd
df = pd.DataFrame(all_posts)
df.to_csv("reddit_posts_raw.csv", index=False)
print(f"\nDataset final guardado: {len(df)} posts totales")
print(f"Periodo: {dates[-1]} a {dates[0]}")
df.head()


🔁 Iniciando extracción de 5 posts/día × 3 subreddits × 365 días


📆 Procesando fechas:   0%|          | 0/365 [00:00<?, ?it/s]

⚠️ Solo se encontraron 0 posts en r/CryptoMarkets el 2025-05-20
⚠️ Solo se encontraron 0 posts en r/CryptoCurrency el 2025-05-20


📆 Procesando fechas:   0%|          | 1/365 [00:02<16:26,  2.71s/it]

⚠️ Solo se encontraron 0 posts en r/CryptoNews el 2025-05-20
⚠️ Solo se encontraron 0 posts en r/CryptoMarkets el 2025-05-19
⚠️ Solo se encontraron 0 posts en r/CryptoCurrency el 2025-05-19


📆 Procesando fechas:   1%|          | 2/365 [00:05<15:34,  2.57s/it]

⚠️ Solo se encontraron 0 posts en r/CryptoNews el 2025-05-19
⚠️ Solo se encontraron 0 posts en r/CryptoMarkets el 2025-05-18
⚠️ Solo se encontraron 0 posts en r/CryptoCurrency el 2025-05-18


📆 Procesando fechas:   1%|          | 3/365 [00:07<15:31,  2.57s/it]

⚠️ Solo se encontraron 0 posts en r/CryptoNews el 2025-05-18


KeyboardInterrupt: 